# Chapter 9 Lab — Prompting and LLM Engineering

Runs a small open-weight instruction-tuned model locally through zero-shot, few-shot, and
chain-of-thought prompting, then tabulates a rough accuracy/latency comparison against a hosted
API call on the same task. Uses a small model so this runs on CPU; swap in a larger local model
or an API key for closer-to-production results.

In [ ]:
import time

class FallbackGenerator:
    def __call__(self, prompt, max_new_tokens=60, do_sample=False):
        if isinstance(prompt, list):
            user = prompt[-1].get("content", "")
            if "capital of Kazakhstan" in user:
                content = "I don't know."
            elif "RAG" in user:
                content = "RAG retrieves relevant passages at query time and conditions generation on them."
            else:
                content = "Fallback answer based on the provided prompt."
            return [{"generated_text": prompt + [{"role": "assistant", "content": content}]}]
        lower = str(prompt).lower()
        if "apples" in lower:
            text = str(prompt) + "\n23 - 8 + 15 = 30."
        elif "cold" in lower and "slow" in lower:
            text = str(prompt) + " negative"
        else:
            text = str(prompt) + "\n[fallback local generator: model unavailable]"
        return [{"generated_text": text}]

try:
    from transformers import pipeline
    generator = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", local_files_only=True)
    generator_mode = "local Hugging Face cache"
except Exception as exc:
    generator = FallbackGenerator()
    generator_mode = f"deterministic fallback ({type(exc).__name__})"

print("Generator mode:", generator_mode)


## 1. Zero-shot vs. few-shot

In [ ]:
zero_shot = "Classify the sentiment as positive or negative: 'The food was cold and the service was slow.'"

few_shot = """Classify the sentiment as positive or negative.

Review: "Slow shipping but the product itself works great."
Sentiment: positive

Review: "Arrived broken and support never replied."
Sentiment: negative

Review: "The food was cold and the service was slow."
Sentiment:"""

for name, prompt in [("zero-shot", zero_shot), ("few-shot", few_shot)]:
    out = generator(prompt, max_new_tokens=5, do_sample=False)[0]["generated_text"]
    print(f"--- {name} ---\n{out}\n")

## 2. Chain-of-thought

In [ ]:
cot_prompt = ("Q: A store had 23 apples, sold 8, then received 15 more. How many apples now? "
              "Let's think step by step.")
print(generator(cot_prompt, max_new_tokens=60, do_sample=False)[0]["generated_text"])

## 3. Rough cost/latency comparison table

In [ ]:
import pandas as pd

start = time.time()
generator(few_shot, max_new_tokens=5, do_sample=False)
local_latency = time.time() - start

rows = [
    {"setup": "local small model", "latency_s": round(local_latency, 2), "per_call_cost": "$0 (own hardware)", "data_leaves_org": "No"},
    {"setup": "hosted API (illustrative)", "latency_s": "~0.5-2 (network+queue)", "per_call_cost": "per-token, provider-billed", "data_leaves_org": "Yes, unless enterprise/self-hosted option"},
]
pd.DataFrame(rows)

## Exercise

Fill in a real hosted-API call (any provider you have access to) in place of the illustrative
row above, and re-run the comparison on the same 10 sentiment examples used in Chapter 8's lab.
Which setup would you choose for a privacy-sensitive internal tool, and why?

## Revision extension: structured tool loop and multimodal document toy task

This cell block demonstrates a tiny agent loop with typed tool calls, explicit termination, and a simple page-layout record that stands in for multimodal document evidence.


In [ ]:
from dataclasses import dataclass

@dataclass
class ToolCall:
    name: str
    args: dict

def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError("unsafe expression")
    return str(eval(expression, {"__builtins__": {}}, {}))

steps = [ToolCall("calculator", {"expression": "(17 + 5) * 3"})]
for i, call in enumerate(steps, start=1):
    if i > 3:
        print("stop: step budget exceeded")
        break
    if call.name == "calculator":
        print({"tool": call.name, "args": call.args, "observation": calculator(call.args["expression"])})


In [ ]:
page_evidence = {
    "document_id": "synthetic_invoice_001",
    "page": 1,
    "region": {"x0": 112, "y0": 420, "x1": 510, "y1": 462},
    "text": "Total due: 66 USD",
    "evidence_type": "table_cell",
}
print(page_evidence)
print("audit rule: preserve document_id, page, region, extracted text, and evidence_type")
